
# 11 Booking Breakdown

## Your Objective

Your dataset contains ~40,000 reservations for a 200-room resort hotel in Portugal, including the booking date, check-in & check-out dates, and a cancellation flag.

Your task is to calculate the resort's occupancy rate for each month in the dataset.

Occupancy rate = booked room-nights ÷ available room-nights

A guest occupies a room every night from their check-in date up to (but not including) their check-out date

Canceled reservations should not count toward occupancy

See sample logic below:

![image](../images/2Rd3cyrVI2Bdr5xBuayqyZw0.avif)

In [9]:
import pandas as pd
import numpy as np

In [11]:
df = pd.read_csv('data/hotel_bookings.csv', parse_dates=['booking_date', 'cancel_date', 'checkin_date', 'checkout_date'])
df.head()

,booking_id,booking_date,cancel_date,checkin_date,checkout_date,is_canceled
0,1,2014-03-18,NaT,2016-02-25,2016-03-24,0
1,2,2014-04-18,NaT,2015-10-02,2015-10-11,0
2,3,2014-04-30,NaT,2015-08-03,2015-08-10,0
3,4,2014-04-30,NaT,2015-08-03,2015-08-10,0
4,5,2014-04-30,NaT,2015-08-03,2015-08-10,0


In [13]:
df = df[df.is_canceled == 0]
df.head()

,booking_id,booking_date,cancel_date,checkin_date,checkout_date,is_canceled
0,1,2014-03-18,NaT,2016-02-25,2016-03-24,0
1,2,2014-04-18,NaT,2015-10-02,2015-10-11,0
2,3,2014-04-30,NaT,2015-08-03,2015-08-10,0
3,4,2014-04-30,NaT,2015-08-03,2015-08-10,0
4,5,2014-04-30,NaT,2015-08-03,2015-08-10,0


In [18]:
df['nights'] = [pd.date_range(checkin, checkout, inclusive='left') for checkin, checkout in zip(df.checkin_date, df.checkout_date)]
df.head()

,booking_id,booking_date,cancel_date,checkin_date,checkout_date,is_canceled,nights
0,1,2014-03-18,NaT,2016-02-25,2016-03-24,0,"DatetimeIndex(['2016-02-25', '2016-02-26', '20..."
1,2,2014-04-18,NaT,2015-10-02,2015-10-11,0,"DatetimeIndex(['2015-10-02', '2015-10-03', '20..."
2,3,2014-04-30,NaT,2015-08-03,2015-08-10,0,"DatetimeIndex(['2015-08-03', '2015-08-04', '20..."
3,4,2014-04-30,NaT,2015-08-03,2015-08-10,0,"DatetimeIndex(['2015-08-03', '2015-08-04', '20..."
4,5,2014-04-30,NaT,2015-08-03,2015-08-10,0,"DatetimeIndex(['2015-08-03', '2015-08-04', '20..."


In [20]:
df_nights = df.explode('nights')
df_nights.head()

,booking_id,booking_date,cancel_date,checkin_date,checkout_date,is_canceled,nights
0,1,2014-03-18,NaT,2016-02-25,2016-03-24,0,2016-02-25
0,1,2014-03-18,NaT,2016-02-25,2016-03-24,0,2016-02-26
0,1,2014-03-18,NaT,2016-02-25,2016-03-24,0,2016-02-27
0,1,2014-03-18,NaT,2016-02-25,2016-03-24,0,2016-02-28
0,1,2014-03-18,NaT,2016-02-25,2016-03-24,0,2016-02-29


In [22]:
df_month = df_nights.nights.dt.to_period('M').value_counts().sort_index().to_frame(name='bookings')
df_month.head()

,bookings
nights,
2015-07,4944
2015-08,5557
2015-09,5428
2015-10,4829
2015-11,3295


In [26]:
df_month['available_rooms'] = 200 * df_month.index.days_in_month
df_month['occupancy_rate'] = (df_month.bookings / df_month.available_rooms) * 100
df_month.head()

,bookings,available_rooms,occupancy_rate
nights,,,
2015-07,4944,6200,79.741935
2015-08,5557,6200,89.629032
2015-09,5428,6000,90.466667
2015-10,4829,6200,77.887097
2015-11,3295,6000,54.916667


In [28]:
df_month.loc['2016-07'].occupancy_rate

np.float64(88.56451612903226)